# Week 7 - Dimensionality Reduction (1/2)
**Notebook 1 : Feature Selection - the 10 metrics of the lecture**

**Unstructured Data Analysis (2026-2)** · Professor: Misuk Kim · Teaching Assistant: Hojin Son

A term-document matrix has one column per word: more than ten thousand columns for 2,000 reviews, almost all zero.
Lecture 6-2 / 7-1 *Dimensionality Reduction* offers two roads: **feature selection** keeps some of the columns
(this notebook), **feature extraction** builds new ones (Notebook 2).

| Lecture chapter | What we do in code |
|---|---|
| 1. Dimensionality Reduction | measure how wide and how sparse the matrix really is |
| 2. Feature Selection, metrics 1-10 | rebuild the lecture's **10 x 10** data set and reproduce its tables (DF, Acc, AccR, PR, OddR, OddN, F1, IG, $\chi^2$, BNS) |
| 2. Feature Selection, applied | the same 10 metrics on 2,000 movie reviews |
| 1. Filter vs Wrapper | top-$k$ words of a metric (filter) vs words chosen by a learning algorithm (wrapper) |

> **How to use this notebook in Colab**: `File ▸ Save a copy in Drive`, then run the cells from top to bottom with
> `Shift + Enter`. Run the **Setup** cell first. The **🤔 Quick check** notes after an output say what to look for in it.

## &nbsp;0. Setup
NLTK (corpus and stop words), NumPy, pandas, SciPy, scikit-learn. All pre-installed in Colab.

In [ ]:
import nltk
for r in ['movie_reviews', 'stopwords']:
    nltk.download(r, quiet=True)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm
from sklearn.feature_extraction.text import CountVectorizer

%config InlineBackend.figure_format = 'retina'
%matplotlib inline
pd.set_option('display.width', 140)
np.set_printoptions(linewidth=140)

## &nbsp;1. How big is the matrix, really?
*Common features of text data* (lecture): two issues of a term-document matrix

* **Issue 1 - high dimensionality** : many more terms than documents
* **Issue 2 - sparseness** : most elements are zero

Measured on the 2,000 NLTK movie reviews (1,000 positive, 1,000 negative), as a **binary** matrix
(1 = the word occurs in the review): every metric of the lecture counts *documents*, not occurrences.

In [ ]:
from nltk.corpus import movie_reviews, stopwords

english_stops = set(stopwords.words('english'))
fileids = movie_reviews.fileids()
docs = [' '.join(movie_reviews.words(f)) for f in fileids]
y = np.array([1 if f.startswith('pos') else 0 for f in fileids])     # 1 = Pos, 0 = Neg

vec = CountVectorizer(binary=True, min_df=5, token_pattern=r'[a-z]{2,}', stop_words=list(english_stops))
X = vec.fit_transform(docs)                 # documents x terms, sparse, entries 0 / 1
terms = vec.get_feature_names_out()

print('documents :', X.shape[0], '  (Pos', (y == 1).sum(), '/ Neg', (y == 0).sum(), ')')
print('terms     :', X.shape[1], '  (words in at least 5 reviews, stop words removed)')
print('non-zero  :', X.nnz, f'of {X.shape[0] * X.shape[1]:,} cells  ->  sparsity {1 - X.nnz / (X.shape[0] * X.shape[1]):.1%}')

**🤔 Quick check** Six times more columns than rows (Issue 1), and 98 % of the cells are empty (Issue 2): a review uses about
240 of the 13,000 words. The lecture's small data set comes next, because on 10 documents every number can be checked by hand.

The lecture's taxonomy (*A simplified taxonomy*):

| | What it does | Uses the label? | Uses the learning algorithm? | Examples |
|---|---|---|---|---|
| **Selection - filter** | keeps columns chosen by a statistic | yes (DF: no) | no | the 10 metrics below |
| **Selection - wrapper** | keeps columns chosen by training a model again and again | yes | yes | forward / backward selection |
| **Extraction** | builds new columns from the old ones | no | no | PCA, MDS, **LSA** (Notebook 2) |

## &nbsp;2. The lecture's artificial data set
*Artificial Data Set*: 10 documents, 10 terms, binary, 6 positive and 4 negative. $A$ has one row per **term** and
one column per **document**, as on the slide.

In [ ]:
A = np.array([
    [1, 1, 1, 1, 1, 1, 0, 0, 0, 0],   # Term 1
    [0, 0, 0, 0, 0, 0, 1, 1, 1, 1],   # Term 2
    [1, 1, 1, 1, 1, 1, 1, 1, 1, 1],   # Term 3
    [1, 1, 1, 1, 1, 1, 1, 1, 0, 0],   # Term 4
    [0, 0, 0, 1, 1, 1, 1, 1, 1, 1],   # Term 5
    [1, 1, 1, 0, 0, 0, 0, 0, 0, 0],   # Term 6
    [0, 0, 0, 0, 0, 0, 1, 1, 0, 0],   # Term 7
    [1, 0, 1, 0, 1, 0, 1, 0, 1, 0],   # Term 8
    [1, 1, 1, 0, 0, 0, 1, 0, 0, 0],   # Term 9
    [1, 0, 0, 0, 0, 0, 0, 0, 1, 1],   # Term 10
])
toy_terms = [f'Term {i}' for i in range(1, 11)]
toy_docs = [f'D{i}' for i in range(1, 11)]
toy_y = np.array([1, 1, 1, 1, 1, 1, 0, 0, 0, 0])      # Pos = 1, Neg = 0

table = pd.DataFrame(A, index=toy_terms, columns=toy_docs)
table.loc['Class'] = ['Pos' if c else 'Neg' for c in toy_y]
table

### &nbsp;2-1. The four counts every metric is made of
* $N(Pos),\ N(Neg)$ : number of positive / negative documents
* $N(Pos, w),\ N(Neg, w)$ : number of positive / negative documents **containing** $w$
* $N(Pos, \bar w),\ N(Neg, \bar w)$ : number of positive / negative documents **not** containing $w$
* $N(w) = N(Pos, w) + N(Neg, w)$, &nbsp; $N = N(Pos) + N(Neg)$

`class_counts` returns them for **every term at once**, from a documents-by-terms matrix, so the same function serves the
10 terms here and the 13,000 terms of Section 3.

In [ ]:
def class_counts(X, y):
    # X: documents x terms (0/1), y: 1 = Pos, 0 = Neg  ->  one value per term
    X = X if isinstance(X, np.ndarray) else X.toarray() if X.shape[1] < 50 else X
    N_pos_w = np.asarray(X[y == 1].sum(axis=0)).ravel().astype(float)
    N_neg_w = np.asarray(X[y == 0].sum(axis=0)).ravel().astype(float)
    return dict(N_pos=float((y == 1).sum()), N_neg=float((y == 0).sum()),
                N_pos_w=N_pos_w, N_neg_w=N_neg_w)


C = class_counts(A.T, toy_y)           # A.T : documents x terms
print('N(Pos) =', C['N_pos'], '  N(Neg) =', C['N_neg'])
print('N(Pos, w) :', C['N_pos_w'])
print('N(Neg, w) :', C['N_neg_w'])

### &nbsp;2-2. Metrics 1-4 : DF, Acc, AccR, PR
$$DF(w) = N_D(w) \qquad Acc(w) = N(Pos,w) - N(Neg,w)$$
$$AccR(w) = \left| \frac{N(Pos,w)}{N(Pos)} - \frac{N(Neg,w)}{N(Neg)} \right| \qquad PR(w) = \frac{N(Pos,w)}{N(Pos)} \bigg/ \frac{N(Neg,w)}{N(Neg)}$$

A word that never occurs in a negative document gives $PR = \infty$, on the slide and in NumPy.

In [ ]:
def metrics_1_to_4(C):
    N_pos, N_neg, N_pos_w, N_neg_w = C['N_pos'], C['N_neg'], C['N_pos_w'], C['N_neg_w']
    with np.errstate(divide='ignore', invalid='ignore'):
        DF = N_pos_w + N_neg_w
        Acc = N_pos_w - N_neg_w
        AccR = np.abs(N_pos_w / N_pos - N_neg_w / N_neg)
        PR = (N_pos_w / N_pos) / (N_neg_w / N_neg)
    return pd.DataFrame({'DF': DF, 'Acc': Acc, 'AccR': AccR, 'PR': PR})


m14 = metrics_1_to_4(C)
m14.index = toy_terms
m14.round(2)

**🤔 Quick check** The slide's table, number for number. Term 3 is in every document: the largest DF, but AccR = 0 and PR = 1.
The most frequent word says nothing about the class, which is the lecture's warning about DF: *"words that appear too often may
provide unnecessary information"*.

### &nbsp;2-3. Metrics 5-7 : OddR, OddN, F1
$$OddR(w) = \frac{N(Pos,w)}{N(Neg,w)} \times \frac{N(Neg,\bar w)}{N(Pos,\bar w)} \qquad OddN(w) = N(Pos,w) \times N(Neg,\bar w)$$
$$F1(w) = \frac{2 \times N(Pos,w)}{N(Pos) + N(w)}$$

OddR, slide rule: *"Add 1 to any zero count in the denominator"*. Only a count that **is** zero becomes 1.

In [ ]:
def metrics_5_to_7(C):
    N_pos, N_neg, N_pos_w, N_neg_w = C['N_pos'], C['N_neg'], C['N_pos_w'], C['N_neg_w']
    N_pos_nw = N_pos - N_pos_w            # N(Pos, not w)
    N_neg_nw = N_neg - N_neg_w            # N(Neg, not w)
    den1 = np.where(N_neg_w == 0, 1, N_neg_w)      # the slide's rule: a zero count in a denominator becomes 1
    den2 = np.where(N_pos_nw == 0, 1, N_pos_nw)
    OddR = (N_pos_w / den1) * (N_neg_nw / den2)
    OddN = N_pos_w * N_neg_nw
    F1 = 2 * N_pos_w / (N_pos + N_pos_w + N_neg_w)
    return pd.DataFrame({'OddR': OddR, 'OddN': OddN, 'F1': F1})


m57 = metrics_5_to_7(C)
m57.index = toy_terms
m57.round(2)

**🤔 Quick check** Term 8: $\frac{3}{2} \times \frac{2}{3} = 1$, Term 9: $\frac{3}{1} \times \frac{3}{3} = 3$, the two worked examples of the slide.
Term 2 and Term 7 live only in negative documents and get **0**: OddR, OddN (and Acc, PR, F1) are **one-sided**, they rank words
for the *positive* class. The lecture says it for F1, *"negative features are devalued compared to positive features"*.

> Term 4: the slide prints $OddR = 4.00$, but by the slide's own rule ($N(Pos,\bar w) = 0 \rightarrow 1$) it is
> $\frac{6}{2} \times \frac{2}{1} = 6$, as printed above. The top-3 count of Section 2-5 is the same either way.

### &nbsp;2-4. Metrics 8-10 : IG, $\chi^2$, BNS
These three use the whole $2 \times 2$ table (class $\times$ present / absent), so they are **symmetric**.

$$IG(w) = Entropy_{absent}(w) - Entropy_{given}(w), \qquad
Entropy_{absent}(w) = \sum_{C} -P(C) \log P(C)$$
$$Entropy_{given}(w) = P(w) \sum_{C} -P(C|w) \log P(C|w) + P(\bar w) \sum_{C} -P(C|\bar w) \log P(C|\bar w)$$

$$\chi^2(w) = \frac{N \left( P(Pos,w) P(Neg,\bar w) - P(Neg,w) P(Pos,\bar w) \right)^2}{P(w) P(\bar w) P(Pos) P(Neg)}$$

$$BNS(w) = \left| F^{-1}\!\left(\frac{N(Pos,w)}{N(Pos)}\right) - F^{-1}\!\left(\frac{N(Neg,w)}{N(Neg)}\right) \right|, \qquad F = \text{c.d.f. of } \mathcal{N}(0, 1)$$

Three conventions of the slides: the logarithm is **base 10** ($-0.6 \log 0.6 - 0.4 \log 0.4 = 0.29$), $0 \log 0 = 0$,
and a rate of 1 (or 0) is replaced by $0.9995$ (or $0.0005$) before $F^{-1}$, since $F^{-1}(1) = \infty$.

In [ ]:
def entropy_terms(p):
    # -p log10 p, elementwise, with 0 log 0 = 0
    p = np.asarray(p, dtype=float)
    out = np.zeros_like(p)
    m = p > 0
    out[m] = -p[m] * np.log10(p[m])
    return out


def metrics_8_to_10(C):
    N_pos, N_neg, N_pos_w, N_neg_w = C['N_pos'], C['N_neg'], C['N_pos_w'], C['N_neg_w']
    N = N_pos + N_neg
    N_w = N_pos_w + N_neg_w
    N_pos_nw, N_neg_nw = N_pos - N_pos_w, N_neg - N_neg_w
    with np.errstate(divide='ignore', invalid='ignore'):
        # --- IG
        P_w, P_nw = N_w / N, 1 - N_w / N
        E_absent = entropy_terms(N_pos / N) + entropy_terms(N_neg / N)
        E_given = (P_w * (entropy_terms(N_pos_w / N_w) + entropy_terms(N_neg_w / N_w))
                   + P_nw * (entropy_terms(N_pos_nw / (N - N_w)) + entropy_terms(N_neg_nw / (N - N_w))))
        IG = E_absent - np.nan_to_num(E_given)
        # --- chi-squared
        num = N * ((N_pos_w / N) * (N_neg_nw / N) - (N_neg_w / N) * (N_pos_nw / N)) ** 2
        den = P_w * P_nw * (N_pos / N) * (N_neg / N)
        chi2 = np.where(den > 0, num / den, 0.0)        # a word in every (or no) document: 0
        # --- BNS
        tpr = np.clip(N_pos_w / N_pos, 0.0005, 0.9995)
        fpr = np.clip(N_neg_w / N_neg, 0.0005, 0.9995)
        BNS = np.abs(norm.ppf(tpr) - norm.ppf(fpr))
    return pd.DataFrame({'IG': IG, 'chi2': chi2, 'BNS': BNS})


m810 = metrics_8_to_10(C)
m810.index = toy_terms
m810.round(2)

**🤔 Quick check** Term 1: $IG = 0.29$ (knowing Term 1 removes all the uncertainty about the class), $\chi^2 = 10$,
$BNS = 3.29 - (-3.29) = 6.58$. Term 4: $BNS = 3.29$, the slide's example. Term 2 scores exactly like Term 1: these three
do not care which class the word points to.

### &nbsp;2-5. All ten metrics, and which words they agree on
The summary slide counts, for every term, in how many of the ten rankings it is in the **top 3**.
Ties share a rank (Term 1 and Term 2 are both first for IG, so the next one is third).

In [ ]:
def all_metrics(X, y, names):
    C = class_counts(X, y)
    M = pd.concat([metrics_1_to_4(C), metrics_5_to_7(C), metrics_8_to_10(C)], axis=1)
    M.index = names
    return M


def top3_count(M, k=3, decimals=2):
    # rank every column from the largest value down; ties share the best rank (as on the slide).
    # decimals=2 reproduces the slide's rounded table; use decimals=None on real data.
    R = M if decimals is None else M.round(decimals)
    ranks = R.rank(ascending=False, method='min')
    return (ranks <= k).sum(axis=1)


M_toy = all_metrics(A.T, toy_y, toy_terms)
summary = M_toy.round(2)
summary['Top3'] = top3_count(M_toy)
summary

**🤔 Quick check** The slide *Comparison of the 10 metrics*: Term 4 is in the top 3 of all ten, Term 1 of nine, Term 6 of six.
Term 2 and Term 7, the words of the negative class, are found by the four symmetric metrics (AccR, IG, $\chi^2$, BNS) and by nothing else.

## &nbsp;3. The same ten metrics on 2,000 reviews
`all_metrics` takes any documents-by-terms matrix: here the sparse matrix of Section 1 (Pos = 1). One line, ten metrics, every word.

In [ ]:
M = all_metrics(X, y, terms)
print(M.shape)
M.loc[['film', 'bad', 'wonderful', 'boring', 'waste', 'perfectly']].round(3)

**🤔 Quick check** `film` plays the role of Term 3 here: it occurs in 1,753 of the 2,000 reviews (top DF), and IG, $\chi^2$ and BNS
are about 0, since positive and negative reviews use it alike. `bad` and `waste` are negative words: Acc < 0, PR < 1, OddR small,
yet IG, $\chi^2$ and BNS large.

In [ ]:
def top_words(M, k=10):
    return pd.DataFrame({col: M[col].sort_values(ascending=False).index[:k] for col in M.columns},
                        index=[f'#{i}' for i in range(1, k + 1)])


top_words(M, 10)

**🤔 Quick check** Three kinds of columns:

* **DF** : the words of the *topic* (film, movie, one) - frequent, useless for the class
* **Acc, PR, OddR, OddN, F1** : *positive* words only. PR and OddR divide by the negative count, so words that are **never**
  negative win however rare they are (`obscurity`: five reviews, $PR = \infty$)
* **AccR, IG, $\chi^2$, BNS** : words that *separate* the classes, in both directions (`bad`, `worst`, `boring` next to `life`, `great`)

**❓ Why is the BNS column full of rare words?** A rate of 0 is clipped to 0.0005, and $F^{-1}(0.0005) = -3.29$ is far below
$F^{-1}(5/1000) = -2.58$. On ten documents the clip never mattered; on two thousand it decides the ranking.

In [ ]:
agreement = top3_count(M, k=20, decimals=None).sort_values(ascending=False)
print(agreement.head(12).to_string())

**🤔 Quick check** No word is in the top 20 of more than six metrics; the lecture's Term 4 (ten out of ten) has no counterpart on
real data. The words most metrics agree on are the frequent positive ones: five metrics (Acc, PR, OddR, OddN, F1) favor the
positive class, and DF rewards frequency, so a frequent positive word scores well on six lists at once.

## &nbsp;4. Filter vs Wrapper
Slide *Filter approach vs. Wrapper approach*:

* **Filter** : rank the words by a metric, keep the top $k$. No feedback loop, independent of the learning algorithm.
* **Wrapper** : evaluate a subset by training a learning algorithm, keep what improves it, repeat.

The learning algorithm is scikit-learn's `LogisticRegression`, used as a **black box** (its own topic in Week 10):
trained on 1,500 reviews, accuracy reported on the other 500.

The test **labels** must stay unseen by everything that chooses words: the metrics of Section 3 used all 2,000 labels,
so they are recomputed here on the 1,500 training reviews only (`M_train`). Selecting words with the test labels would
leak the answer into the test score. (The vocabulary of Section 1, `min_df=5`, was built on all 2,000 reviews; that uses
word presence, not labels, and is left as it is for simplicity. A strict pipeline would fit the vectorizer on the
training reviews too.)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

train_idx, test_idx = train_test_split(np.arange(X.shape[0]), test_size=500, stratify=y, random_state=42)
M_train = all_metrics(X[train_idx], y[train_idx], terms)      # metrics from the training labels only


def score(columns):
    # accuracy of a logistic regression that sees only the given columns (a black box, see Week 10)
    cols = np.asarray(columns)
    clf = LogisticRegression(max_iter=2000)
    clf.fit(X[train_idx][:, cols], y[train_idx])
    return clf.score(X[test_idx][:, cols], y[test_idx])


print(f'all {X.shape[1]} words : accuracy {score(np.arange(X.shape[1])):.3f}')

### &nbsp;4-1. Filter : the top-$k$ words of a metric

In [ ]:
ks = [20, 100, 500, 2000, X.shape[1]]
rows = {}
for metric in ['DF', 'PR', 'F1', 'IG', 'chi2', 'BNS']:
    order = M_train[metric].sort_values(ascending=False).index
    rows[metric] = [score([vec.vocabulary_[w] for w in order[:k]]) for k in ks]

filter_table = pd.DataFrame(rows, index=[f'top {k}' for k in ks[:-1]] + ['all']).T
filter_table.round(3)

**🤔 Quick check** IG row: 2,000 words, 15 % of the columns, give nearly the accuracy of all 13,037 (0.83 vs 0.84), and 100
words already give 0.81. PR row: the same number of words, rare and purely positive, is far worse; DF's frequent words are worse
too. The ranking matters, not the count. That is the lecture's promise: fewer features without substantially sacrificing
accuracy. (Last digits can differ between scikit-learn versions.)

**❓ Why `M_train` and not `M`?** With the metrics of Section 3 (all 2,000 labels) the IG row reads 0.84 / 0.87 / 0.89: 500 words
would seem to *beat* all 13,037. That gain is leakage: the test labels helped choose the words. Feature selection is part of
training, and must see training data only.

> scikit-learn's `SelectKBest(chi2)` is **not** the lecture's $\chi^2$: it only compares the counts of the word's *presence*
> and ignores the absent-row of the $2 \times 2$ table. The assignment uses the lecture's formula, as above.

### &nbsp;4-2. Wrapper : let the learning algorithm choose
Greedy **forward selection**: start from no word, try every candidate, keep the one that raises the accuracy most, repeat.
Candidates: the 100 best $\chi^2$ words; 10 words are chosen. The wrapper never looks at the test reviews while choosing:
it chooses on a validation split cut from the training reviews, and the candidates themselves are ranked with the labels
of the remaining 1,200 training reviews only (`M_tr`). Cost: about 1,000 model fits, a few seconds.

In [ ]:
tr_idx, val_idx = train_test_split(train_idx, test_size=300, stratify=y[train_idx], random_state=42)
M_tr = all_metrics(X[tr_idx], y[tr_idx], terms)                # candidates ranked without the validation labels


def val_score(columns):
    cols = np.asarray(columns)
    clf = LogisticRegression(max_iter=2000)
    clf.fit(X[tr_idx][:, cols], y[tr_idx])
    return clf.score(X[val_idx][:, cols], y[val_idx])


candidates = [vec.vocabulary_[w] for w in M_tr['chi2'].sort_values(ascending=False).index[:100]]
chosen, curve = [], []
for step in range(10):
    best = max((c for c in candidates if c not in chosen), key=lambda c: val_score(chosen + [c]))
    chosen.append(best)
    curve.append(val_score(chosen))
    print(f'step {step + 1:>2}: + {terms[best]:<12} validation accuracy {curve[-1]:.3f}')

filter_words = list(M_train['chi2'].sort_values(ascending=False).index[:10])    # the filter of 4-1, ten words
filter_cols = [vec.vocabulary_[w] for w in filter_words]
print('\nwrapper, 10 words :', [terms[c] for c in chosen])
print('filter,  10 words :', filter_words)
print(f'\ntest accuracy  wrapper {score(chosen):.3f}   filter top-10 chi2 {score(filter_cols):.3f}')

**🤔 Quick check** About a thousand trainings for ten words, against one sort for the filter. The wrapper's ten are not the ten
best $\chi^2$ words: once `bad` is in, a second word that says the same thing (`worst`, `stupid`) adds less than a word that
says something new (`overall`, `realistic`). That feedback loop is the whole difference between the two approaches.

**❓ Why is the wrapper not clearly better on the test reviews?** It chose the words that fit its 300 validation reviews, and a thousand fits bought nothing the filter did not have. The lecture's
*"can offer higher accuracy but may take longer"* is a *can*; the *longer* is certain.

---
### ✅ What we practiced
* A term-document matrix is wide and empty: 13,000 columns for 2,000 reviews, 98 % zeros
* Ten metrics, ten ways of reading four counts: DF ignores the class; Acc, PR, OddR, OddN, F1 favor the positive class; AccR, IG, $\chi^2$, BNS are symmetric
* Filter = rank and cut, once. Wrapper = train, compare, repeat. A good filter with a few hundred words loses nothing, and the selection must use training labels only
* Notebook 2 takes the other road: new columns with SVD (LSA), drawn with t-SNE